# Vì sao Direction DL dao động theo seed nhiều hơn XGBoost? Chồng lấp chuỗi hay phương sai do khởi tạo?

**Giả thuyết A (chồng lấp):** LSTM đọc 24 hàng liên tiếp, nên các mẫu chồng lấp 23/24 hàng; seed dao động vì chuỗi.
→ Nếu đúng: chuỗi càng ngắn thì các seed càng giống nhau, và LSTM 1 nến phải ổn định gần như XGBoost.

**Giả thuyết B (khởi tạo / tối ưu):** khởi tạo ngẫu nhiên, thứ tự batch và dropout, cộng với dừng sớm ở epoch 1–2 khi tín hiệu yếu.
→ Nếu đúng: độ ổn định gần như không đổi theo độ dài chuỗi. TabM (1 hàng, nhưng có sẵn ensemble 16 thành viên bên trong) ổn định hơn LSTM.

**So sánh trên cùng 3 seed (1000–1002), touch luôn là XGBoost, chỉ đổi model Direction:**
- XGBoost, TabM (1 hàng);
- LSTM với chuỗi 1 / 6 / 12 / 24 / 48 nến.

**Đo theo từng cặp seed:**
1. Tương quan Spearman của p_up trên mọi giờ test (và dev).
2. Jaccard của tập giờ "tự tin": top 10% |p_up − 0.5| mỗi quý. Đây là phần quyết định lệnh.
3. Giao dịch với tham số đã khoá: TB / std / seed tệ nhất của R/lệnh.

**Chi phí và thứ tự chạy:**
- XGBoost, TabM, LSTM 24 lấy từ cache.
- LSTM 12 và 48 lấy từ cache của `exp_lstm_overfit` mục 6, nên chạy notebook đó xong trước.
- LSTM 1 và 6 phải train mới: 20 quý × 3 seed mỗi biến thể, khoảng 30–60 phút trên GPU, có cache.

Kết quả lưu ở `artifacts/1h_strategy_v2/exp_lstm_overfit/seed_variance*.csv`.

In [1]:
import os, sys
from dataclasses import replace
from itertools import combinations
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "model_training").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "model_training"))
import numpy as np
import pandas as pd
import strategy_h1 as st

pd.set_option("display.width", 220, "display.max_columns", 40)
cfg = st.load_locked("artifacts/1h_strategy_v2/config_locked.json")
XG, TB, LS = (replace(cfg, model_kind=k, nn_params={}) for k in ("xgb", "tabm", "lstm"))
OUT = Path(cfg.output_dir) / "exp_lstm_overfit"
OUT.mkdir(parents=True, exist_ok=True)
D = st.build(cfg)
SEEDS3 = cfg.seeds[:3]

def keep_seeds(Pf, seeds=SEEDS3):
    drop = [c for c in Pf.columns if "_s" in c and c.split("_s")[-1].isdigit() and int(c.split("_s")[-1]) not in seeds]
    Q = Pf.drop(columns=drop).copy()
    for c in ("p_touch", "p_up"):
        Q[c] = Q[[f"{c}_s{s}" for s in seeds]].mean(axis=1)
    return Q

PX = {ph: st.walk_forward(D, XG, ph)[0] for ph in ("dev", "test")}            # cache
M = {}
for ph in ("dev", "test"):
    M.setdefault("XGBoost (1 hàng)", {})[ph] = keep_seeds(PX[ph])
    Pt = st.walk_forward(D, TB, ph)[0]                                         # cache compare_models
    M.setdefault("TabM (1 hàng, ensemble 16 bên trong)", {})[ph] = keep_seeds(st.blend_direction({"xgb": PX[ph], "tabm": Pt}, {"tabm": 1.0}))
    Pl = st.walk_forward(D, LS, ph)[0]                                         # cache: LSTM 24
    M.setdefault("LSTM chuỗi 24", {})[ph] = keep_seeds(st.blend_direction({"xgb": PX[ph], "lstm": Pl}, {"lstm": 1.0}))
for L in (1, 6, 12, 48):                                                       # 12 / 48: cache mục 6; 1 / 6: train mới
    for ph in ("dev", "test"):
        M.setdefault(f"LSTM chuỗi {L}", {})[ph] = keep_seeds(
            st.walk_forward_direction(D, replace(LS, nn_params={"seq_len": L}), PX[ph], ph, seeds=SEEDS3))
list(M)

Dữ liệu: 54,979 nến 1h, 53,522 mẫu (2020-08-30 -> 2026-10-08), touch 92 feature, direction 37 feature, 11s
dev 2022Q1: AUC touch 0.610 | AUC direction 0.616 | train 7,303 | 0s
dev 2022Q2: AUC touch 0.695 | AUC direction 0.536 | train 9,511 | 0s
dev 2022Q3: AUC touch 0.605 | AUC direction 0.520 | train 11,719 | 0s
dev 2022Q4: AUC touch 0.689 | AUC direction 0.601 | train 13,879 | 0s
dev 2023Q1: AUC touch 0.664 | AUC direction 0.675 | train 16,063 | 0s
dev 2023Q2: AUC touch 0.692 | AUC direction 0.588 | train 18,271 | 0s
dev 2023Q3: AUC touch 0.751 | AUC direction 0.565 | train 20,479 | 0s
dev 2023Q4: AUC touch 0.721 | AUC direction 0.566 | train 22,639 | 0s
dev 2024Q1: AUC touch 0.718 | AUC direction 0.571 | train 24,823 | 0s
dev 2024Q2: AUC touch 0.749 | AUC direction 0.587 | train 27,031 | 0s
dev 2024Q3: AUC touch 0.735 | AUC direction 0.585 | train 29,239 | 0s
dev 2024Q4: AUC touch 0.778 | AUC direction 0.626 | train 31,423 | 0s
test 2025Q1: AUC touch 0.763 | AUC direction 0.552 | tr

['XGBoost (1 hàng)',
 'TabM (1 hàng, ensemble 16 bên trong)',
 'LSTM chuỗi 24',
 'LSTM chuỗi 1',
 'LSTM chuỗi 6',
 'LSTM chuỗi 12',
 'LSTM chuỗi 48']

## 1. Độ giống nhau giữa các seed

In [2]:
def tail_sets(Pf, col, frac=0.10):
    T = Pf[Pf.is_test]
    d = (T[col] - 0.5).abs()
    thr = d.groupby(T.fold).transform(lambda x: x.quantile(1 - frac))
    return set(T.t_ns[d >= thr])

rows = []
for name, PP in M.items():
    for ph in ("dev", "test"):
        T = PP[ph][PP[ph].is_test]
        cols = [f"p_up_s{s}" for s in SEEDS3]
        C = T[cols].corr(method="spearman").to_numpy()
        cor = C[np.triu_indices(len(cols), 1)]
        sets = [tail_sets(PP[ph], c) for c in cols]
        jac = [len(a & b) / len(a | b) for a, b in combinations(sets, 2)]
        rows.append({"model": name, "phase": ph, "tương quan seed TB": cor.mean(), "tương quan seed thấp nhất": cor.min(),
                     "Jaccard top 10% TB": np.mean(jac), "Jaccard top 10% thấp nhất": np.min(jac)})
SV = pd.DataFrame(rows)
SV.to_csv(OUT / "seed_variance_agreement.csv", index=False)
SV.pivot(index="model", columns="phase", values=["tương quan seed TB", "Jaccard top 10% TB", "Jaccard top 10% thấp nhất"]).round(3)

tương quan seed TB        Jaccard top 10% TB        Jaccard top 10% thấp nhất       
phase                                               dev   test                dev   test                       dev   test
model                                                                                                                    
LSTM chuỗi 1                                      0.893  0.885              0.546  0.518                     0.534  0.483
LSTM chuỗi 12                                     0.755  0.830              0.444  0.420                     0.431  0.382
LSTM chuỗi 24                                     0.750  0.819              0.444  0.452                     0.388  0.427
LSTM chuỗi 48                                     0.755  0.777              0.455  0.411                     0.413  0.379
LSTM chuỗi 6                                      0.763  0.824              0.444  0.437                     0.429  0.422
TabM (1 hàng, ensemble 16 bên trong)              0.864  0.865              0.497  0.519                     0.467  0.482
XGBoost (1 hàng)                                  0.977  0.986              0.746  0.788                     0.744  0.778

## 2. Giao dịch với tham số đã khoá (touch XGBoost, chỉ đổi Direction; 3 seed)

In [3]:
rows = []
for name, PP in M.items():
    for ph in ("dev", "test"):
        S = st.seed_table(D, PP[ph], cfg, ph)
        rows.append({"model": name, "phase": ph, "R ens": S.loc["ens", "R/lệnh"], "TB seed": S.loc["TB seed", "R/lệnh"],
                     "std seed": S.loc["std seed", "R/lệnh"], "seed tệ nhất": S.loc["seed tệ nhất", "R/lệnh"],
                     "AUC chạm trước": st.direction_auc(D, PP[ph])["AUC chạm trước (đã chạm)"].mean()})
SR = pd.DataFrame(rows)
SR.to_csv(OUT / "seed_variance_trading.csv", index=False)
SR.pivot(index="model", columns="phase", values=["AUC chạm trước", "R ens", "TB seed", "std seed", "seed tệ nhất"]).round(3)

AUC chạm trước         R ens        TB seed        std seed        seed tệ nhất       
phase                                           dev   test    dev   test     dev   test      dev   test          dev   test
model                                                                                                                      
LSTM chuỗi 1                                  0.592  0.566  0.194  0.123   0.185  0.114    0.020  0.012        0.162  0.101
LSTM chuỗi 12                                 0.591  0.576  0.193  0.195   0.154  0.186    0.046  0.026        0.103  0.157
LSTM chuỗi 24                                 0.592  0.565  0.189  0.186   0.162  0.209    0.047  0.056        0.112  0.144
LSTM chuỗi 48                                 0.594  0.573  0.220  0.172   0.171  0.189    0.037  0.076        0.129  0.105
LSTM chuỗi 6                                  0.588  0.572  0.212  0.166   0.181  0.148    0.025  0.024        0.163  0.130
TabM (1 hàng, ensemble 16 bên trong)          0.589  0.578  0.196  0.154   0.195  0.133    0.007  0.028        0.187  0.111
XGBoost (1 hàng)                              0.585  0.571  0.156  0.185   0.162  0.166    0.016  0.028        0.145  0.140

**Cách đọc:**
- Tương quan và Jaccard của LSTM tăng rõ khi chuỗi ngắn lại, và LSTM 1 nến gần bằng XGBoost: chồng lấp chuỗi là nguyên nhân (giả thuyết A).
- Chúng gần như không đổi theo độ dài chuỗi, và TabM ổn định hơn mọi LSTM: phương sai đến từ khởi tạo / tối ưu (giả thuyết B). Khi đó cách giảm phương sai là thêm seed hoặc ensemble bên trong model, không phải rút ngắn chuỗi.
- Các cột R/lệnh chỉ để tham khảo, không dùng để chọn độ dài chuỗi, vì đây là bảng nhìn cả test.